# Message Queues — the simulations behind "System Design, Plainly — Part 4"

Four toy simulations, all runnable here. They are small on purpose: the numbers in the post
come from these cells (seeded, so they reproduce exactly), plus the cited external benchmarks
flagged in the post text.


## 1. Synchronous chain vs async queue

A checkout touches 5 services in sequence, each 40 ms, each failing independently with p = 1%.
Synchronously, one failure fails the user. Asynchronously, the user waits only for the 2 ms
enqueue; a downstream failure is retried from the queue.


In [ ]:
import random
import numpy as np

rng = random.Random(20261108)
N, N_SERVICES, SVC_MS, P_FAIL = 100_000, 5, 40, 0.01

sync_fail = 0
latencies = []
for _ in range(N):
    lat, failed = 0, False
    for _ in range(N_SERVICES):
        lat += SVC_MS
        if rng.random() < P_FAIL:
            failed = True
            break
    latencies.append(lat)
    sync_fail += failed

print(f"sync chain: user-visible failure = {sync_fail / N:.2%} "
      f"(theory 1 - 0.99**5 = {1 - 0.99**5:.2%})")
print(f"sync chain: p50 latency = {np.percentile(latencies, 50):.0f} ms, "
      f"full chain = {N_SERVICES * SVC_MS} ms")
print(f"async queue: user-facing latency = 2 ms enqueue; "
      f"user-visible failure = 0% (failures retried from the queue)")
print(f"latency improvement: {N_SERVICES * SVC_MS / 2:.0f}x")

assert abs(sync_fail / N - (1 - 0.99**5)) < 0.005
assert np.percentile(latencies, 50) == 200


## 2. Delivery semantics: at-most-once, at-least-once, exactly-once

10,000 messages; the consumer crashes mid-processing with p = 1% per message.
At-most-once acks *before* processing, so a crash loses the message.
At-least-once acks *after* processing, so a crash redelivers it (duplicate).
Exactly-once = at-least-once delivery + an idempotent consumer with a dedup store.


In [ ]:
N_MSG, P_CRASH = 10_000, 0.01
lost = dup = 0
for _ in range(N_MSG):
    if rng.random() < P_CRASH:
        lost += 1   # at-most-once: gone
        dup += 1    # at-least-once: redelivered -> processed twice

print(f"at-most-once : {lost} lost ({lost / N_MSG:.2%}), 0 duplicates")
print(f"at-least-once: 0 lost, {dup} duplicates ({dup / N_MSG:.2%})")
print(f"exactly-once : 0 lost, 0 duplicates (idempotent consumer + dedup store)")
print(f"theory: expected lost/dup = N * p = {N_MSG * P_CRASH:.0f}")

assert 50 < lost < 150 and 50 < dup < 150  # ~100 expected, binomial spread


## 3. Consumer lag and backpressure (Little's law)

Producer bursts at 12,000 msg/s for 60 s, then drops to 4,000 msg/s. Consumer holds 8,000 msg/s.
Backlog grows at (arrival − service) and drains at (service − arrival). Little's law:
backlog = arrival rate × wait, so the last message in a 240,000-deep queue waits 30 s.
Second run: the consumer is down for 30 s mid-burst.


In [ ]:
PROD_BURST, PROD_BASE, CONS = 12_000, 4_000, 8_000
T_BURST, T_END = 60, 150

def run(outage=None):
    backlog, peak = 0.0, 0.0
    for t in range(T_END + 1):
        if t == 0:
            continue
        prod = PROD_BURST if t <= T_BURST else PROD_BASE
        cons = 0 if (outage and outage[0] <= t < outage[1]) else CONS
        backlog = max(0.0, backlog + (prod - cons))
        peak = max(peak, backlog)
    return peak

peak = run()
peak_outage = run(outage=(20, 50))
drain_t = T_BURST + peak / (CONS - PROD_BASE)

print(f"healthy consumer : peak backlog = {peak:,.0f} msgs "
      f"(theory (12000-8000)*60 = {(PROD_BURST - CONS) * T_BURST:,})")
print(f"                   drain completes at t = {drain_t:.0f} s")
print(f"Little's law     : 240,000 / 8,000 per s = {240_000 / CONS:.0f} s "
      f"wait for the last message in line")
print(f"30 s outage      : peak backlog = {peak_outage:,.0f} msgs (2x the healthy peak)")

assert peak == 240_000
assert drain_t == 120
assert peak_outage == 480_000


## 4. Ordering vs parallelism: the partition tradeoff

One partition = strict order, one consumer at a time: 5,000 msg/s.
Each added partition adds a parallel consumer, but global order is gone — ordering holds
only *within* a partition (same key → same partition). A hot key (30% of traffic) is
indivisible: the busiest partition floors at 30% no matter how many partitions you add.


In [ ]:
PER_PART = 5_000
HOT = 0.30

for p in (1, 2, 4, 8, 16, 32):
    thr = p * PER_PART
    # busiest partition: the hot key (30%) lands on one partition, which also
    # receives its even share of the remaining 70%
    busiest = (HOT + (1 - HOT) / p) * 100
    order = "strict global order" if p == 1 else "order within partition only"
    print(f"{p:2d} partitions: {thr:,} msg/s | busiest partition {busiest:4.1f}% "
          f"(ideal {100 / p:4.1f}%) | {order}")

assert 32 * PER_PART == 160_000
assert abs((HOT + (1 - HOT) / 32) - 0.3219) < 0.001  # busiest floors near the hot key itself


## 5. Same-hardware broker benchmark (April 2026, ComputingForGeeks)

Kafka 4.2.0 (KRaft) vs Redpanda 26.1.2 community edition, identical 4 vCPU / 8 GB VMs,
Kafka's own perf-test tools, 5M messages sustained. Note the honest caveat: other
benchmarks (including Redpanda's own) report the opposite ranking on p99 latency —
vendor numbers disagree, so treat any single benchmark as one data point, not a verdict.


In [ ]:
KAFKA_SUST, RP_SUST = 211_283, 89_455      # records/s, sustained 5M messages
KAFKA_P99, RP_P99 = 41, 514              # ms, p99 end-to-end latency
KAFKA_AVG, RP_AVG = 2.53, 315.42         # ms, average latency

print(f"sustained throughput: Kafka {KAFKA_SUST:,} rec/s vs Redpanda {RP_SUST:,} "
      f"-> Kafka {KAFKA_SUST / RP_SUST:.1f}x")
print(f"p99 latency         : Kafka {KAFKA_P99} ms vs Redpanda {RP_P99} ms "
      f"-> Kafka {RP_P99 / KAFKA_P99:.1f}x lower")
print(f"avg latency         : Kafka {KAFKA_AVG} ms vs Redpanda {RP_AVG} ms "
      f"-> Kafka {RP_AVG / KAFKA_AVG:.0f}x lower")
print("caveat: vendor benchmarks report the opposite p99 ranking — "
      "benchmarks disagree, load-test your own workload")

assert abs(KAFKA_SUST / RP_SUST - 2.36) < 0.05
assert abs(RP_P99 / KAFKA_P99 - 12.5) < 0.1
